In [1]:
import pandas as pd
df = pd.read_csv("../data/emails.csv")
print(len(df))
df.head()

517401


,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


In [3]:
print(df["message"][0])

Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 16:39:00 -0700 (PDT)
From: phillip.allen@enron.com
To: tim.belden@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Tim Belden <Tim Belden/Enron@EnronXGate>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Here is our forecast

 


In [5]:
import re

def parse_email(raw):

    fields = {}

    patterns = {
        "message_id": r"Message-ID:\s*(.*)",
        "date": r"Date:\s*(.*)",
        "from": r"From:\s*(.*)",
        "to": r"To:\s*(.*)",
        "subject": r"Subject:\s*(.*)"
    }

    for key, pattern in patterns.items():
        match = re.search(pattern, raw)
        fields[key] = match.group(1).strip() if match else ""

    return fields

In [6]:
parsed = df["message"].apply(parse_email)
meta_df = pd.json_normalize(parsed)

meta_df.head()

,message_id,date,from,to,subject
0,<18782981.1075855378110.JavaMail.evans@thyme>,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",phillip.allen@enron.com,tim.belden@enron.com,Mime-Version: 1.0
1,<15464986.1075855378456.JavaMail.evans@thyme>,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",phillip.allen@enron.com,john.lavorato@enron.com,Re:
2,<24216240.1075855687451.JavaMail.evans@thyme>,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test
3,<13505866.1075863688222.JavaMail.evans@thyme>,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",phillip.allen@enron.com,randall.gay@enron.com,Mime-Version: 1.0
4,<30922949.1075863688243.JavaMail.evans@thyme>,"Thu, 31 Aug 2000 05:07:00 -0700 (PDT)",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello


In [9]:
import re

def extract_body(raw):
    
    # split after the last header line
    body_start = re.split(r"\nX-FileName:.*\n", raw)

    if len(body_start) > 1:
        return body_start[1].strip()

    # fallback method
    parts = raw.split("\n\n", 1)
    if len(parts) > 1:
        return parts[1].strip()

    return ""

In [10]:
df["body"] = df["message"].apply(extract_body)

In [11]:
df[["subject", "body"]].head()

KeyError: "['subject'] not in index"

In [12]:
print(df.columns)

Index(['file', 'message', 'body'], dtype='object')


In [15]:
print(df["message"].iloc[3])

Message-ID: <13505866.1075863688222.JavaMail.evans@thyme>
Date: Mon, 23 Oct 2000 06:13:00 -0700 (PDT)
From: phillip.allen@enron.com
To: randall.gay@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Randall L Gay
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Dec2000\Notes Folders\'sent mail
X-Origin: Allen-P
X-FileName: pallen.nsf

Randy,

 Can you send me a schedule of the salary and level of everyone in the 
scheduling group.  Plus your thoughts on any changes that need to be made.  
(Patti S for example)

Phillip


In [12]:
import re

def parse_email(raw):

    data = {}

    fields = [
        "Message-ID",
        "Date",
        "From",
        "To",
        "Subject",
        "X-From",
        "X-To",
        "X-Folder"
    ]

    for field in fields:
        match = re.search(rf"{field}:\s*(.*)", raw)
        data[field.lower().replace("-", "_")] = match.group(1).strip() if match else ""

    # Extract body
    parts = raw.split("\n\n", 1)
    body = parts[1] if len(parts) > 1 else ""

    data["body"] = body.strip()

    return data

In [14]:
parsed = df["message"].apply(parse_email)
email_df = pd.json_normalize(parsed)

email_df.head()

,message_id,date,from,to,subject,x_from,x_to,x_folder,body
0,<18782981.1075855378110.JavaMail.evans@thyme>,"Mon, 14 May 2001 16:39:00 -0700 (PDT)",phillip.allen@enron.com,tim.belden@enron.com,Mime-Version: 1.0,Phillip K Allen,Tim Belden <Tim Belden/Enron@EnronXGate>,"\Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Se...",Here is our forecast
1,<15464986.1075855378456.JavaMail.evans@thyme>,"Fri, 4 May 2001 13:51:00 -0700 (PDT)",phillip.allen@enron.com,john.lavorato@enron.com,Re:,Phillip K Allen,John J Lavorato <John J Lavorato/ENRON@enronXg...,"\Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Se...",Traveling to have a business meeting takes the...
2,<24216240.1075855687451.JavaMail.evans@thyme>,"Wed, 18 Oct 2000 03:00:00 -0700 (PDT)",phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,Phillip K Allen,Leah Van Arsdall,\Phillip_Allen_Dec2000\Notes Folders\'sent mail,test successful. way to go!!!
3,<13505866.1075863688222.JavaMail.evans@thyme>,"Mon, 23 Oct 2000 06:13:00 -0700 (PDT)",phillip.allen@enron.com,randall.gay@enron.com,Mime-Version: 1.0,Phillip K Allen,Randall L Gay,\Phillip_Allen_Dec2000\Notes Folders\'sent mail,"Randy,\n\n Can you send me a schedule of the s..."
4,<30922949.1075863688243.JavaMail.evans@thyme>,"Thu, 31 Aug 2000 05:07:00 -0700 (PDT)",phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Phillip K Allen,Greg Piper,\Phillip_Allen_Dec2000\Notes Folders\'sent mail,Let's shoot for Tuesday at 11:45.


In [15]:
df_clean = pd.concat([df, email_df], axis=1)

In [19]:
df_clean[["from", "to", "subject", "body"]].head()

,from,to,subject,body,body
0,phillip.allen@enron.com,tim.belden@enron.com,Mime-Version: 1.0,Here is our forecast,Here is our forecast
1,phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...,Traveling to have a business meeting takes the...
2,phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!,test successful. way to go!!!
3,phillip.allen@enron.com,randall.gay@enron.com,Mime-Version: 1.0,"Randy,\n\n Can you send me a schedule of the s...","Randy,\n\n Can you send me a schedule of the s..."
4,phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.,Let's shoot for Tuesday at 11:45.


In [16]:
df_clean = df_clean.loc[:, ~df_clean.columns.duplicated()]

In [17]:
df_clean.columns

Index(['file', 'message', 'message_id', 'date', 'from', 'to', 'subject',
       'x_from', 'x_to', 'x_folder', 'body'],
      dtype='object')

In [18]:
df_clean[["from", "to", "subject", "body"]].head()

,from,to,subject,body
0,phillip.allen@enron.com,tim.belden@enron.com,Mime-Version: 1.0,Here is our forecast
1,phillip.allen@enron.com,john.lavorato@enron.com,Re:,Traveling to have a business meeting takes the...
2,phillip.allen@enron.com,leah.arsdall@enron.com,Re: test,test successful. way to go!!!
3,phillip.allen@enron.com,randall.gay@enron.com,Mime-Version: 1.0,"Randy,\n\n Can you send me a schedule of the s..."
4,phillip.allen@enron.com,greg.piper@enron.com,Re: Hello,Let's shoot for Tuesday at 11:45.


In [23]:
def extract_subject(raw):

    for line in raw.split("\n"):
        if line.startswith("Subject:"):
            return line.replace("Subject:", "").strip()

    return ""

In [24]:
df["subject"] = df["message"].apply(extract_subject)

In [25]:
print(df_clean.columns)

Index(['file', 'message', 'body', 'message_id', 'date', 'from', 'to',
       'subject', 'x_from', 'x_to', 'x_folder'],
      dtype='object')


In [19]:
print(df_clean["body"].isna().sum())

0


In [20]:
print(df_clean["subject"].value_counts().head())

subject
Mime-Version: 1.0                                        17015
RE:                                                       6477
Re:                                                       6308
Demand Ken Lay Donate Proceeds from Enron Stock Sales     1124
FW:                                                        938
Name: count, dtype: int64


In [21]:
import re

def normalize_subject(subject):

    if not isinstance(subject, str):
        return ""

    subject = subject.lower().strip()

    # remove reply prefixes repeatedly
    subject = re.sub(r'^(re|fw|fwd)\s*:\s*', '', subject)

    # remove extra spaces
    subject = re.sub(r'\s+', ' ', subject)

    return subject.strip()

In [22]:
df_clean["subject_clean"] = df_clean["subject"].apply(normalize_subject)

In [23]:
df_clean[["subject", "subject_clean"]].head(10)

,subject,subject_clean
0,Mime-Version: 1.0,mime-version: 1.0
1,Re:,
2,Re: test,test
3,Mime-Version: 1.0,mime-version: 1.0
4,Re: Hello,hello
5,Re: Hello,hello
6,Mime-Version: 1.0,mime-version: 1.0
7,Re: PRC review - phone calls,prc review - phone calls
8,Re: High Speed Internet Access,high speed internet access
9,FW: fixed forward or other Collar floor gas pr...,fixed forward or other collar floor gas price ...


In [24]:
df_clean = df_clean[df_clean["subject_clean"] != ""]

In [25]:
df_clean = df_clean[~df_clean["subject_clean"].isin(["re", "test"])]

In [26]:
df_clean[["subject", "subject_clean"]].head(10)

,subject,subject_clean
0,Mime-Version: 1.0,mime-version: 1.0
3,Mime-Version: 1.0,mime-version: 1.0
4,Re: Hello,hello
5,Re: Hello,hello
6,Mime-Version: 1.0,mime-version: 1.0
7,Re: PRC review - phone calls,prc review - phone calls
8,Re: High Speed Internet Access,high speed internet access
9,FW: fixed forward or other Collar floor gas pr...,fixed forward or other collar floor gas price ...
10,Re: FW: fixed forward or other Collar floor ga...,fw: fixed forward or other collar floor gas pr...
11,Mime-Version: 1.0,mime-version: 1.0


In [27]:
threads = df_clean.groupby("subject_clean")
thread_sizes = threads.size().sort_values(ascending=False)

In [35]:
thread_sizes.head(20)

subject_clean
mime-version: 1.0                                        17015
demand ken lay donate proceeds from enron stock sales     1124
schedule crawler: hourahead failure                        900
enron mentions                                             835
schedule crawler: hourahead failure <codesite>             800
entouch newsletter                                         554
(no subject)                                               544
lunch                                                      531
hey                                                        458
meeting                                                    453
hi                                                         443
organizational announcement                                431
organizational changes                                     416
congratulations                                            415
apb checkout                                               387
energy issues                            

In [28]:
valid_threads = thread_sizes[thread_sizes >= 3]

In [29]:
selected_subjects = valid_threads.head(15).index

dataset = df_clean[df_clean["subject_clean"].isin(selected_subjects)]

In [30]:
thread_map = {
    subject: f"T-{i:03d}"
    for i, subject in enumerate(selected_subjects, start=1)
}

dataset["thread_id"] = dataset["subject_clean"].map(thread_map)

/var/folders/40/70k0cdxd1871nw_c8jz6t_7r0000gn/T/ipykernel_50964/4095952198.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset["thread_id"] = dataset["subject_clean"].map(thread_map)


In [31]:
print("Threads:", dataset["thread_id"].nunique())
print("Messages:", len(dataset))

Threads: 15
Messages: 25306


In [32]:
df_clean["date"].min(), df_clean["date"].max()

('Fri, 1 Aug 1997 01:00:00 -0700 (PDT)',
 'Wed, 9 Oct 2002 14:20:21 -0700 (PDT)')

In [41]:
df_clean["month"] = df_clean["date"].dt.to_period("M")

df_clean["month"].value_counts().sort_index()

AttributeError: Can only use .dt accessor with datetimelike values

In [42]:
df_clean["date"] = pd.to_datetime(df_clean["date"], errors="coerce")

/var/folders/40/70k0cdxd1871nw_c8jz6t_7r0000gn/T/ipykernel_27771/2544305112.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_clean["date"] = pd.to_datetime(df_clean["date"], errors="coerce")
/var/folders/40/70k0cdxd1871nw_c8jz6t_7r0000gn/T/ipykernel_27771/2544305112.py:1: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df_clean["date"] = pd.to_datetime(df_clean["date"], errors="coerce")


In [43]:
df_clean["date"].head()

0    2001-05-14 16:39:00-07:00
3    2000-10-23 06:13:00-07:00
4    2000-08-31 05:07:00-07:00
5    2000-08-31 04:17:00-07:00
6    2000-08-22 07:44:00-07:00
Name: date, dtype: object

In [44]:
df_clean["date"].dtype

dtype('O')

In [45]:
df_clean = df_clean.dropna(subset=["date"])

In [46]:
df_clean["month"] = df_clean["date"].dt.to_period("M")

AttributeError: Can only use .dt accessor with datetimelike values

In [47]:
print(df_clean["date"].dtype)

object


In [48]:
import pandas as pd

df_clean["date"] = pd.to_datetime(df_clean["date"], errors="coerce")

In [49]:
print(df_clean["date"].dtype)

datetime64[ns, tzoffset('PDT', -25200)]


In [50]:
df_clean["date"].head()

0   2001-05-14 16:39:00-07:00
3   2000-10-23 06:13:00-07:00
4   2000-08-31 05:07:00-07:00
5   2000-08-31 04:17:00-07:00
6   2000-08-22 07:44:00-07:00
Name: date, dtype: datetime64[ns, tzoffset('PDT', -25200)]

In [51]:
df_clean = df_clean.dropna(subset=["date"])

In [52]:
df_clean["month"] = df_clean["date"].dt.to_period("M")

/var/folders/40/70k0cdxd1871nw_c8jz6t_7r0000gn/T/ipykernel_27771/3631880068.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_clean["month"] = df_clean["date"].dt.to_period("M")


In [53]:
df_clean["month"].value_counts().sort_index()

month
1986-05        1
1997-04       36
1997-05       32
1997-06       64
1997-07       56
1997-08       77
1997-09       72
1997-10       24
1998-05        1
1998-09        1
1999-04       79
1999-05      656
1999-06      631
1999-07      851
1999-08     1041
1999-09     1231
1999-10     1382
2000-04     8470
2000-05    10114
2000-06    13505
2000-07    13263
2000-08    18559
2000-09    19372
2000-10    21445
2001-04    35333
2001-05    34836
2001-06    18236
2001-07    10036
2001-08     8607
2001-09    10560
2001-10    30985
2002-04      890
2002-05      900
2002-06      901
2002-07      244
2002-09        6
2002-10        1
2024-05        1
Freq: M, Name: count, dtype: int64

In [54]:
df_clean["month"].value_counts().sort_index()

month
1986-05        1
1997-04       36
1997-05       32
1997-06       64
1997-07       56
1997-08       77
1997-09       72
1997-10       24
1998-05        1
1998-09        1
1999-04       79
1999-05      656
1999-06      631
1999-07      851
1999-08     1041
1999-09     1231
1999-10     1382
2000-04     8470
2000-05    10114
2000-06    13505
2000-07    13263
2000-08    18559
2000-09    19372
2000-10    21445
2001-04    35333
2001-05    34836
2001-06    18236
2001-07    10036
2001-08     8607
2001-09    10560
2001-10    30985
2002-04      890
2002-05      900
2002-06      901
2002-07      244
2002-09        6
2002-10        1
2024-05        1
Freq: M, Name: count, dtype: int64

In [55]:
print("Total emails:", len(df_clean))
print("Date range:", df_clean["date"].min(), "→", df_clean["date"].max())

Total emails: 262499
Date range: 1986-05-01 07:37:34-07:00 → 2024-05-26 03:49:57-07:00


In [56]:
df_clean["month"] = df_clean["date"].dt.to_period("M")

df_clean["month"].value_counts().sort_index()


/var/folders/40/70k0cdxd1871nw_c8jz6t_7r0000gn/T/ipykernel_27771/2910912031.py:1: UserWarning: Converting to PeriodArray/Index representation will drop timezone information.
  df_clean["month"] = df_clean["date"].dt.to_period("M")


month
1986-05        1
1997-04       36
1997-05       32
1997-06       64
1997-07       56
1997-08       77
1997-09       72
1997-10       24
1998-05        1
1998-09        1
1999-04       79
1999-05      656
1999-06      631
1999-07      851
1999-08     1041
1999-09     1231
1999-10     1382
2000-04     8470
2000-05    10114
2000-06    13505
2000-07    13263
2000-08    18559
2000-09    19372
2000-10    21445
2001-04    35333
2001-05    34836
2001-06    18236
2001-07    10036
2001-08     8607
2001-09    10560
2001-10    30985
2002-04      890
2002-05      900
2002-06      901
2002-07      244
2002-09        6
2002-10        1
2024-05        1
Freq: M, Name: count, dtype: int64

In [57]:
start = "2000-10-01"
end = "2001-03-31"

df_window = df_clean[
    (df_clean["date"] >= start) &
    (df_clean["date"] <= end)
]

In [58]:
print("Messages in window:", len(df_window))

Messages in window: 21445


In [59]:
thread_sizes = (
    df_window.groupby("subject_clean")
    .size()
    .sort_values(ascending=False)
)

In [60]:
thread_sizes.head(20)

subject_clean
mime-version: 1.0                                             960
a computer and internet connection for you and your family     71
lunch                                                          62
signature authority                                            61
global operations controller forum                             57
cc: don.miller@enron.com                                       57
(no subject)                                                   56
questions                                                      50
enron mentions                                                 49
associate/analyst super saturday participation                 48
organization announcement                                      47
enron year end 2000 performance management process             44
open enrollment 2001                                           43
vacation                                                       42
teams                                                         

In [61]:
valid_threads = thread_sizes[thread_sizes >= 3]

In [62]:
selected_threads = []
message_count = 0

for subject, size in valid_threads.items():
    selected_threads.append(subject)
    message_count += size

    if message_count >= 200:
        break

In [63]:
dataset = df_window[
    df_window["subject_clean"].isin(selected_threads)
]

In [64]:
thread_map = {
    subject: f"T-{i:03d}"
    for i, subject in enumerate(selected_threads, start=1)
}

dataset["thread_id"] = dataset["subject_clean"].map(thread_map)

/var/folders/40/70k0cdxd1871nw_c8jz6t_7r0000gn/T/ipykernel_27771/525701965.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset["thread_id"] = dataset["subject_clean"].map(thread_map)


In [65]:
print("Threads:", dataset["thread_id"].nunique())
print("Messages:", len(dataset))
print("Date range:", dataset["date"].min(), "→", dataset["date"].max())

Threads: 1
Messages: 960
Date range: 2000-10-01 04:16:00-07:00 → 2000-10-28 09:29:00-07:00


In [66]:
len(dataset)

960

In [67]:
dataset["thread_id"].nunique()

1

In [68]:
df_clean["subject"].value_counts().head(20)

subject
Mime-Version: 1.0                                                   9324
Schedule Crawler: HourAhead Failure <CODESITE>                       719
Enron Mentions                                                       607
EnTouch Newsletter                                                   289
Mid-Year 2001 Performance Feedback                                   273
Energy Issues                                                        222
Organization Announcement                                            220
Organizational Announcement                                          184
All-Employee Meeting                                                 182
Williams Energy News Live -- today's video newscast                  148
Re: (no subject)                                                     140
(no subject)                                                         132
Yahoo! Breaking News                                                 126
3 - URGENT - TO PREVENT LOSS OF INFORMATION

In [69]:
df_clean["subject_clean"].value_counts().head(20)

subject_clean
mime-version: 1.0                                      9324
schedule crawler: hourahead failure <codesite>          719
enron mentions                                          640
entouch newsletter                                      305
(no subject)                                            300
meeting                                                 291
lunch                                                   291
mid-year 2001 performance feedback                      283
confidentiality agreement                               250
energy issues                                           228
organization announcement                               225
organizational announcement                             214
vacation                                                189
all-employee meeting                                    186
hey                                                     179
hi                                                      174
thank you                 

In [70]:
df_clean["subject_clean"].nunique()

70773

In [71]:
df_clean["thread_key"] = (
    df_clean["subject_clean"].fillna("") +
    "_" +
    df_clean["from"].fillna("") +
    "_" +
    df_clean["to"].fillna("")
)

In [72]:
thread_sizes = df_clean.groupby("thread_key").size().sort_values(ascending=False)

thread_sizes.head(20)

thread_key
schedule crawler: hourahead failure <codesite>_pete.davis@enron.com_pete.davis@enron.com                                                      719
mime-version: 1.0_vince.kaminski@enron.com_vkaminski@aol.com                                                                                  286
enron mentions_ann.schmidt@enron.com_X-cc:                                                                                                    250
enron mentions_m..schmidt@enron.com_X-cc:                                                                                                     249
mime-version: 1.0_chris.germany@enron.com_8777208398.4891940@pagenetmessage.net                                                               130
yahoo! breaking news_alerts-breakingnews@yahoo-inc.com_mike.grigsby@enron.com                                                                 126
entouch newsletter_enron.announcements@enron.com_all_ena_egm_eim@enron.com                                       

In [73]:
valid_threads = thread_sizes[thread_sizes >= 3]

In [74]:
len(valid_threads)

34194

In [75]:
selected_threads = []
message_count = 0

for thread, size in valid_threads.items():
    selected_threads.append(thread)
    message_count += size
    
    if message_count >= 200:
        break

In [76]:
dataset = df_clean[df_clean["thread_key"].isin(selected_threads)]

In [77]:
thread_map = {
    key: f"T-{i:03d}"
    for i, key in enumerate(selected_threads, start=1)
}

dataset["thread_id"] = dataset["thread_key"].map(thread_map)

/var/folders/40/70k0cdxd1871nw_c8jz6t_7r0000gn/T/ipykernel_27771/2823891640.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset["thread_id"] = dataset["thread_key"].map(thread_map)


In [78]:
print("Threads:", dataset["thread_id"].nunique())
print("Messages:", len(dataset))

Threads: 1
Messages: 719


In [79]:
df_clean["subject"].nunique()


87013

In [80]:
df_clean["subject_clean"].nunique()

70773

In [81]:
thread_sizes = (
    df_clean.groupby("subject_clean")
    .size()
    .sort_values(ascending=False)
)

thread_sizes.head(20)

subject_clean
mime-version: 1.0                                      9324
schedule crawler: hourahead failure <codesite>          719
enron mentions                                          640
entouch newsletter                                      305
(no subject)                                            300
meeting                                                 291
lunch                                                   291
mid-year 2001 performance feedback                      283
confidentiality agreement                               250
energy issues                                           228
organization announcement                               225
organizational announcement                             214
vacation                                                189
all-employee meeting                                    186
hey                                                     179
hi                                                      174
thank you                 

In [82]:
(thread_sizes >= 3).sum()

np.int64(30160)

In [83]:
selected_threads = []
message_count = 0

for subject, size in thread_sizes.items():

    if size < 3:
        continue

    selected_threads.append(subject)
    message_count += size

    if message_count >= 200:
        break

In [84]:
len(selected_threads)

1

In [85]:
thread_sizes = (
    df_clean.groupby("subject_clean")
    .size()
    .sort_values(ascending=False)
)

thread_sizes.head(20)

subject_clean
mime-version: 1.0                                      9324
schedule crawler: hourahead failure <codesite>          719
enron mentions                                          640
entouch newsletter                                      305
(no subject)                                            300
meeting                                                 291
lunch                                                   291
mid-year 2001 performance feedback                      283
confidentiality agreement                               250
energy issues                                           228
organization announcement                               225
organizational announcement                             214
vacation                                                189
all-employee meeting                                    186
hey                                                     179
hi                                                      174
thank you                 

In [86]:
(thread_sizes >= 2).sum()

np.int64(46392)

In [87]:
selected_threads = []
message_count = 0

for subject, size in thread_sizes.items():

    if size < 2:
        continue

    selected_threads.append(subject)
    message_count += size

    if message_count >= 200:
        break

In [88]:
len(selected_threads)

1

In [89]:
len(dataset)

719

In [90]:
dataset["thread_id"].nunique()

1

In [91]:
size_mb = dataset["body"].str.len().sum() / (1024 * 1024)
print(size_mb, "MB")

1.6788091659545898 MB


In [ ]:
dataset.to_json(
    "data/threaded_emails.json",
    orient="records",
    indent=2
)

: 

In [ ]:
import os
os.makedirs("data", exist_ok=True)

In [1]:
os.listdir()

NameError: name 'os' is not defined

In [33]:
import os
os.makedirs("data", exist_ok=True)

In [34]:
os.listdir()

['dataset_cleaning.ipynb', 'data']

In [35]:
df_clean.columns

Index(['file', 'message', 'message_id', 'date', 'from', 'to', 'subject',
       'x_from', 'x_to', 'x_folder', 'body', 'subject_clean'],
      dtype='object')

In [5]:
def extract_subject(raw):
    for line in raw.split("\n"):
        if line.startswith("Subject:"):
            return line.replace("Subject:", "").strip()
    return ""

df_clean["subject"] = df_clean["message"].apply(extract_subject)

NameError: name 'df_clean' is not defined

In [6]:
import pandas as pd

df = pd.read_csv("../data/emails.csv")


In [7]:
df_clean = df.copy()

In [8]:
df_clean.head()

,file,message
0,allen-p/_sent_mail/1.,Message-ID: <18782981.1075855378110.JavaMail.e...
1,allen-p/_sent_mail/10.,Message-ID: <15464986.1075855378456.JavaMail.e...
2,allen-p/_sent_mail/100.,Message-ID: <24216240.1075855687451.JavaMail.e...
3,allen-p/_sent_mail/1000.,Message-ID: <13505866.1075863688222.JavaMail.e...
4,allen-p/_sent_mail/1001.,Message-ID: <30922949.1075863688243.JavaMail.e...


In [9]:
type(df_clean)

pandas.core.frame.DataFrame

In [10]:
def extract_subject(raw):
    for line in raw.split("\n"):
        if line.startswith("Subject:"):
            return line.replace("Subject:", "").strip()
    return ""

df_clean["subject"] = df_clean["message"].apply(extract_subject)

In [11]:
print(df["message"][0])

Message-ID: <18782981.1075855378110.JavaMail.evans@thyme>
Date: Mon, 14 May 2001 16:39:00 -0700 (PDT)
From: phillip.allen@enron.com
To: tim.belden@enron.com
Subject: 
Mime-Version: 1.0
Content-Type: text/plain; charset=us-ascii
Content-Transfer-Encoding: 7bit
X-From: Phillip K Allen
X-To: Tim Belden <Tim Belden/Enron@EnronXGate>
X-cc: 
X-bcc: 
X-Folder: \Phillip_Allen_Jan2002_1\Allen, Phillip K.\'Sent Mail
X-Origin: Allen-P
X-FileName: pallen (Non-Privileged).pst

Here is our forecast

 


In [36]:
df_clean["month"] = df_clean["date"].dt.to_period("M")

AttributeError: Can only use .dt accessor with datetimelike values

In [37]:
dataset["date"] = dataset["date"].astype(str)

dataset.to_json(
    "data/threaded_emails.json",
    orient="records",
    indent=2
)

/var/folders/40/70k0cdxd1871nw_c8jz6t_7r0000gn/T/ipykernel_50964/2793483501.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  dataset["date"] = dataset["date"].astype(str)
